# ChakraModel Universal Evaluation Harness
**ViT-Large vs PraNet ResNet-101 — Full Stress Mode**

- Zero hardcoded dataset paths (auto-discovers all `/kaggle/input/*` datasets)
- Aggressive batch ramp (starts at 4 → ramps to 128, backs off ONLY at >98% VRAM)
- ViT-Large on cuda:0 | PraNet on cuda:1 (or parallel streams on single GPU)
- FP16 autocast + torch.compile (PyTorch 2.0+)
- Provenance-stamped JSON output — no ghost metrics
- PolypGen2021-Video C1–C6 fix applied
- YOLO conf=0.50 for video (keeps FPR < 10%)

In [ ]:
# ── Install deps ──────────────────────────────────────────────────────────────
import subprocess, sys
pkgs = ['pynvml', 'psutil', 'timm', 'ultralytics']
for pkg in pkgs:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', pkg])
print('Dependencies installed.')

In [ ]:
# ── Environment check ─────────────────────────────────────────────────────────
import torch, os
print(f'PyTorch : {torch.__version__}')
print(f'CUDA    : {torch.version.cuda}')
print(f'GPUs    : {torch.cuda.device_count()}')
for i in range(torch.cuda.device_count()):
    p = torch.cuda.get_device_properties(i)
    print(f'  cuda:{i} = {p.name}  ({p.total_memory/1024**3:.1f} GB)')

# Maximum performance settings
torch.backends.cudnn.benchmark    = True
torch.backends.cudnn.deterministic = False
if hasattr(torch, 'set_float32_matmul_precision'):
    torch.set_float32_matmul_precision('high')
print('\nPerformance settings: cudnn.benchmark=True | TF32=high | deterministic=False')

In [ ]:
# ── Mount source code from Kaggle datasets ────────────────────────────────────
# The harness files are distributed via the attached code dataset.
# Dynamically find them — no hardcoded paths.
import sys
from pathlib import Path

input_root = Path('/kaggle/input')
harness_candidates = list(input_root.rglob('run_eval.py'))
if not harness_candidates:
    raise RuntimeError('Could not find run_eval.py in any attached dataset!')

harness_dir = harness_candidates[0].parent
print(f'Harness found at: {harness_dir}')

# Add harness + src to path
for p in [str(harness_dir), str(harness_dir.parent / 'src')]:
    if p not in sys.path:
        sys.path.insert(0, p)

# Find weight files dynamically
def find_weight(name_pattern: str) -> Path:
    matches = list(input_root.rglob(name_pattern))
    if not matches:
        raise FileNotFoundError(f'Weight file not found: {name_pattern}')
    return matches[0]

VIT_WEIGHTS    = find_weight('chakra_transformer_best.pth')
PRANET_WEIGHTS = find_weight('combo1_best.pth')
print(f'ViT weights   : {VIT_WEIGHTS}')
print(f'PraNet weights: {PRANET_WEIGHTS}')

In [ ]:
# ── Start ResourceMonitor ─────────────────────────────────────────────────────
from resource_monitor import ResourceMonitor
from pathlib import Path

OUTPUT_DIR = Path('/kaggle/working/eval_results')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

monitor = ResourceMonitor(log_dir=OUTPUT_DIR / 'logs', initial_batch=4)
monitor.start()
print('Resource monitor started — aggressive ramp mode active.')

In [ ]:
# ── Discover ALL datasets dynamically ─────────────────────────────────────────
from dataset_discovery import discover, DatasetMeta, DatasetKind

print('Scanning /kaggle/input for all datasets...')
datasets = discover('/kaggle/input')

# PolypGen2021-Video C1-C6 expansion fix (from master_plan.md §5)
POLYPGEN_CENTERS = 6
expanded = []
for ds in datasets:
    if (ds.root / 'images_C1').exists():
        print(f'  [PolypGen Fix] Expanding {ds.name} into per-center subsets...')
        for c in range(1, POLYPGEN_CENTERS + 1):
            img_dir  = ds.root / f'images_C{c}'
            mask_dir = ds.root / f'masks_C{c}'
            if img_dir.exists():
                imgs  = sorted(img_dir.rglob('*.jpg')) + sorted(img_dir.rglob('*.png'))
                masks = sorted(mask_dir.rglob('*.jpg')) + sorted(mask_dir.rglob('*.png')) if mask_dir.exists() else []
                if imgs:
                    expanded.append(DatasetMeta(
                        name=f'{ds.name}_C{c}', root=img_dir,
                        kind=DatasetKind.PAIRED_IMAGE if masks else DatasetKind.IMAGE_ONLY,
                        img_paths=imgs, mask_paths=masks, frame_count=len(imgs)
                    ))
                    print(f'    → C{c}: {len(imgs)} frames, masks={bool(masks)}')
    else:
        expanded.append(ds)

datasets = expanded
total_frames = sum(d.frame_count for d in datasets)
print(f'\nTotal datasets : {len(datasets)}')
print(f'Total frames   : {total_frames:,}')

In [ ]:
# ── Load both models ──────────────────────────────────────────────────────────
from eval_engine import load_models

vit, pranet = load_models(
    vit_ckpt    = VIT_WEIGHTS,
    pranet_ckpt = PRANET_WEIGHTS,
)
print('Both models loaded.')

In [ ]:
# ── CUDA warm-up (saturate caches before timing begins) ───────────────────────
print('Warming up CUDA kernels (5 passes each)...')
dummy = torch.zeros(1, 3, 384, 384, device='cuda:0')
with torch.no_grad(), torch.autocast('cuda', dtype=torch.float16):
    for _ in range(5): _ = vit(dummy)
torch.cuda.synchronize()
if torch.cuda.device_count() >= 2:
    dummy2 = dummy.to('cuda:1')
    with torch.no_grad(), torch.autocast('cuda', dtype=torch.float16):
        for _ in range(5): _ = pranet(dummy2)
    torch.cuda.synchronize('cuda:1')
else:
    with torch.no_grad(), torch.autocast('cuda', dtype=torch.float16):
        for _ in range(5): _ = pranet(dummy)
    torch.cuda.synchronize()
print('Warm-up complete. Timing starts now.')

In [ ]:
# ── RUN EVALUATION ────────────────────────────────────────────────────────────
from eval_engine import run_full_evaluation

try:
    results_path = run_full_evaluation(
        datasets   = datasets,
        vit        = vit,
        pranet     = pranet,
        monitor    = monitor,
        output_dir = OUTPUT_DIR,
        threshold  = 0.45,
    )
    print(f'\n✅ Results saved to: {results_path}')
finally:
    monitor.stop()

In [ ]:
# ── RESULTS SUMMARY TABLE ─────────────────────────────────────────────────────
import json
import pandas as pd

with open(results_path) as f:
    all_res = json.load(f)

rows = []
for r in all_res:
    if 'error' in r:
        rows.append({'Dataset': r['dataset'], 'Error': r['error']})
        continue
    rows.append({
        'Dataset'        : r['dataset'],
        'Frames'         : r['frame_count'],
        'ViT Dice'       : round(r['vit_large']['dice'], 4),
        'ViT B-IoU'      : round(r['vit_large']['boundary_iou'], 4),
        'ViT FPS'        : round(r['vit_large']['fps_avg'], 1),
        'PraNet Dice'    : round(r['pranet']['dice'], 4),
        'PraNet B-IoU'   : round(r['pranet']['boundary_iou'], 4),
        'PraNet FPS'     : round(r['pranet']['fps_avg'], 1),
    })

df = pd.DataFrame(rows)
print(df.to_string(index=False))

# Save CSV alongside JSON
csv_path = results_path.with_suffix('.csv')
df.to_csv(csv_path, index=False)
print(f'\nCSV saved: {csv_path}')